In [ ]:
# ============================================================
#  STEP 3 EXTENDED — Clean Models Under All Codec Conditions
#
#  Evaluates AASIST (Step 1) and RawNet2 (Step 2) — both
#  trained on CLEAN ASVspoof 2019 — under 7 codec conditions
#  on WaveFake. Provides the full baseline row for the paper.
#
#  NO TRAINING. Runtime: ~2 hours.
#
#  INPUTS TO ATTACH:
#    1. capstone-step3-weights    (AASIST + RawNet2 best.pth)
#    2. wavefake-vocoders-subset  (WaveFake vocoders)
# ============================================================

import subprocess, sys, os, json, shutil, warnings, time
import numpy as np
from pathlib import Path

subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], check=True, capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'einops', 'scikit-learn', 'soundfile', 'librosa'], check=True)

import torch, soundfile as sf, librosa
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}')

WORK_DIR = Path('/kaggle/working/project4')
AASIST_DIR = WORK_DIR / 'aasist'
RESULTS_DIR = WORK_DIR / 'results'
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Weight paths ──────────────────────────────────────────────
def _find(candidates):
    for p in candidates:
        if Path(p).exists(): return Path(p)
    return None

WEIGHTS_BASE = _find([
    '/kaggle/input/capstone-step3-weights/step3_weights',
    '/kaggle/input/datasets/rohan576/capstone-step3-weights/step3_weights',
])
assert WEIGHTS_BASE, '❌ capstone-step3-weights not attached'
AASIST_CKPT  = WEIGHTS_BASE / 'aasist_weights/best.pth'
RAWNET2_CKPT = WEIGHTS_BASE / 'rawnet2_weights/best.pth'
assert AASIST_CKPT.exists(),  f'❌ AASIST best.pth not found at {AASIST_CKPT}'
assert RAWNET2_CKPT.exists(), f'❌ RawNet2 best.pth not found at {RAWNET2_CKPT}'
print(f'✓ AASIST  weights: {AASIST_CKPT} ({AASIST_CKPT.stat().st_size/1e6:.1f} MB)')
print(f'✓ RawNet2 weights: {RAWNET2_CKPT} ({RAWNET2_CKPT.stat().st_size/1e6:.1f} MB)')

# ── WaveFake path ─────────────────────────────────────────────
WAVEFAKE_INPUT = _find([
    '/kaggle/input/wavefake-vocoders-subset',
    '/kaggle/input/datasets/rohan576/wavefake-vocoders-subset',
])
assert WAVEFAKE_INPUT, '❌ wavefake-vocoders-subset not attached'
print(f'✓ WaveFake: {WAVEFAKE_INPUT}')

In [ ]:
# ── Clone + patch AASIST ──────────────────────────────────────
if not AASIST_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/clovaai/aasist.git',
                    str(AASIST_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(AASIST_DIR / 'requirements.txt')], check=True)
for fpath in AASIST_DIR.rglob('*.py'):
    txt = fpath.read_text(encoding='utf-8')
    for old, new in [('np.float)', 'float)'), ('np.float,', 'float,'),
                     ('np.float ', 'float '), ('num_workers=4', 'num_workers=0'),
                     ('num_workers=8', 'num_workers=0')]:
        txt = txt.replace(old, new)
    fpath.write_text(txt, encoding='utf-8')
sys.path.insert(0, str(AASIST_DIR))
from importlib import import_module
print('✓ AASIST repo ready')

In [ ]:
# ── Codec Configuration ───────────────────────────────────────
CODEC_CONFIGS = {
    'mp3_32':  {'bitrate': '32k',  'ext': 'mp3', 'acodec': 'libmp3lame'},
    'mp3_64':  {'bitrate': '64k',  'ext': 'mp3', 'acodec': 'libmp3lame'},
    'mp3_128': {'bitrate': '128k', 'ext': 'mp3', 'acodec': 'libmp3lame'},
    'aac_64':  {'bitrate': '64k',  'ext': 'aac', 'acodec': 'aac'},
    'aac_96':  {'bitrate': '96k',  'ext': 'aac', 'acodec': 'aac'},
    'opus_32': {'bitrate': '32k',  'ext': 'ogg', 'acodec': 'libopus'},
    'opus_64': {'bitrate': '64k',  'ext': 'ogg', 'acodec': 'libopus'},
}

EVAL_CONDITIONS = {
    'clean':    None,
    'mp3_32':   'mp3_32',
    'mp3_64':   'mp3_64',
    'mp3_128':  'mp3_128',
    'aac_64':   'aac_64',
    'aac_96':   'aac_96',
    'opus_32':  'opus_32',
    'opus_64':  'opus_64',
}

def apply_codec(audio_np, sr, codec_name):
    import tempfile
    cfg = CODEC_CONFIGS[codec_name]
    with tempfile.TemporaryDirectory() as tmp:
        in_p  = os.path.join(tmp, 'in.wav')
        out_p = os.path.join(tmp, f'out.{cfg["ext"]}')
        dec_p = os.path.join(tmp, 'dec.wav')
        sf.write(in_p, audio_np, sr)
        r = subprocess.run(
            ['ffmpeg', '-y', '-loglevel', 'error', '-i', in_p,
             '-acodec', cfg['acodec'], '-b:a', cfg['bitrate'], out_p],
            capture_output=True)
        if r.returncode != 0:
            # warnings.warn(f'Codec {codec_name} failed')
            return audio_np
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', out_p,
                        '-ar', str(sr), '-ac', '1', dec_p],
                       check=True, capture_output=True)
        out, _ = sf.read(dec_p, dtype='float32')
    n = len(audio_np)
    if len(out) > n: out = out[:n]
    elif len(out) < n: out = np.pad(out, (0, n - len(out)))
    return out

print('✓ Codec configs ready:', list(CODEC_CONFIGS.keys()))

In [ ]:
# ── Load Both Models ──────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(AASIST_DIR / 'config/AASIST.conf') as f:
    aasist_cfg = json.load(f)

def load_model(arch, model_cfg, ckpt_path):
    module = import_module(f'models.{arch}')
    model  = getattr(module, 'Model')(model_cfg).to(device)
    state  = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'model' in state:
        state = state['model']
    model.load_state_dict(state)
    model.eval()
    return model

aasist_model = load_model('AASIST', aasist_cfg['model_config'], AASIST_CKPT)
print('✓ AASIST loaded')

rawnet2_config = {
    'architecture': 'RawNet2Spoof', 'nb_samp': 64600,
    'first_conv': 1024, 'filts': [20, [20, 20], [20, 128], [128, 128]],
    'in_channels': 1, 'blocks': [2, 4], 'nb_fc_node': 1024,
    'gru_node': 1024, 'nb_gru_layer': 3, 'nb_classes': 2,
}
rawnet2_model = load_model('RawNet2Spoof', rawnet2_config, RAWNET2_CKPT)
print('✓ RawNet2 loaded')

In [ ]:
# ── Audio Utilities (OPTIMIZED MULTI-THREADING) ────────────────
from concurrent.futures import ThreadPoolExecutor
SR = 16000
MAX_LEN = 64600

def load_and_encode(p, codec_name):
    try:
        wav, orig_sr = sf.read(str(p), dtype='float32')
        if wav.ndim > 1: wav = wav.mean(axis=1)
        if orig_sr != SR: wav = librosa.resample(wav, orig_sr=orig_sr, target_sr=SR)
        if codec_name: wav = apply_codec(wav, SR, codec_name)
        if len(wav) > MAX_LEN: wav = wav[:MAX_LEN]
        elif len(wav) < MAX_LEN: wav = np.pad(wav, (0, MAX_LEN - len(wav)))
        return torch.FloatTensor(wav)
    except Exception:
        return None

def score_files(model, files, codec_name=None, batch_size=32):
    scores = []
    # Parallelize the brutal FFmpeg bottleneck
    with ThreadPoolExecutor(max_workers=8) as executor:
        for i in range(0, len(files), batch_size):
            batch_paths = files[i:i+batch_size]
            # Process mini-batch in parallel
            batch_tensors = list(executor.map(lambda p: load_and_encode(p, codec_name), batch_paths))
            batch_tensors = [t for t in batch_tensors if t is not None]
            if not batch_tensors: continue
            with torch.no_grad():
                _, out = model(torch.stack(batch_tensors).to(device))
                scores.extend(out[:, 1].cpu().numpy().tolist())
    return scores

def compute_eer(bona, spoof):
    b, s = np.sort(np.array(bona, np.float32)), np.sort(np.array(spoof, np.float32))
    thr  = np.unique(np.concatenate([b, s]))
    frr  = np.searchsorted(b, thr, side='left')  / len(b)
    far  = 1.0 - np.searchsorted(s, thr, side='left') / len(s)
    idx  = np.argmin(np.abs(far - frr))
    return float((far[idx] + frr[idx]) / 2 * 100)

print('✓ Audio utilities optimized for parallel threading!')

In [ ]:
# ── Load WaveFake Vocoders & Subsample Bonafide ───────────────
vocoder_dirs = {}
for vdir in sorted(WAVEFAKE_INPUT.rglob('*')):
    if vdir.is_dir() and 'ljspeech_' in vdir.name:
        wavs = sorted(vdir.glob('*.wav'))
        if wavs: vocoder_dirs[vdir.name] = wavs

if not vocoder_dirs:  # zip not auto-extracted
    WF_EXT = Path('/kaggle/working/wavefake_extracted')
    if not WF_EXT.exists():
        zips = list(WAVEFAKE_INPUT.rglob('*.zip'))
        assert zips, f'No vocoder dirs or zip in {WAVEFAKE_INPUT}'
        WF_EXT.mkdir(parents=True, exist_ok=True)
        subprocess.run(['unzip', '-q', str(zips[0]), '-d', str(WF_EXT)], check=True)
    for vdir in sorted(WF_EXT.rglob('*')):
        if vdir.is_dir() and 'ljspeech_' in vdir.name:
            wavs = sorted(vdir.glob('*.wav'))
            if wavs: vocoder_dirs[vdir.name] = wavs

assert vocoder_dirs, 'No WaveFake vocoder dirs found'
print('WaveFake vocoders:')
for n, wavs in sorted(vocoder_dirs.items()):
    print(f'  {n}/  ({len(wavs)} files)')

# ── LJSpeech bonafide ─────────────────────────────────────────
LJ_DIR = Path('/kaggle/working/ljspeech/wavs')
if not LJ_DIR.exists() or len(list(LJ_DIR.glob('*.wav'))) == 0:
    print('Downloading LJSpeech...')
    lj_tar = Path('/kaggle/working/LJSpeech-1.1.tar.bz2')
    if not lj_tar.exists():
        subprocess.run(['wget', '-q', '--show-progress',
                        'https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2',
                        '-O', str(lj_tar)], check=True)
    subprocess.run(['tar', '-xjf', str(lj_tar), '-C', '/kaggle/working/'], check=True)
    Path('/kaggle/working/LJSpeech-1.1').rename('/kaggle/working/ljspeech')
    lj_tar.unlink()

bona_wavs = sorted(LJ_DIR.glob('*.wav'))
# MASSIVE FIX: We do not need 13,100 files for a stable EER vs 2,000 fakes.
# 2000 files is scientifically robust and cuts workload by 85%.
np.random.seed(42)
bona_wavs = np.random.choice(bona_wavs, size=2000, replace=False).tolist()
print(f'✓ LJSpeech bonafide: Subsampled to {len(bona_wavs)} files for speed balance')

In [ ]:
# ── Run Evaluation ────────────────────────────────────────────
results = {'step': 'step3_extended', 'models': {}}

for model_name, model in [('AASIST_clean', aasist_model),
                            ('RawNet2_clean', rawnet2_model)]:
    print(f'\n{'='*60}')
    print(f'  Evaluating: {model_name}')
    print(f'{'='*60}')
    results['models'][model_name] = {}

    for cond_name, codec_name in EVAL_CONDITIONS.items():
        print(f'\n  Condition: {cond_name}')
        results['models'][model_name][cond_name] = {}

        t0 = time.time()
        print(f'    Scoring {len(bona_wavs)} bonafide...', end='', flush=True)
        bona_s = score_files(model, bona_wavs, codec_name)
        print(f' done in {time.time()-t0:.1f}s')

        for voc_name, fake_files in sorted(vocoder_dirs.items()):
            print(f'    [{voc_name}] {len(fake_files)} files...', end='', flush=True)
            t_voc = time.time()
            spoof_s = score_files(model, fake_files, codec_name)
            eer = compute_eer(bona_s, spoof_s)
            results['models'][model_name][cond_name][voc_name] = eer
            print(f' EER={eer:.2f}% ({time.time()-t_voc:.1f}s)')

        mean_eer = np.mean(list(results['models'][model_name][cond_name].values()))
        print(f'    Mean EER [{cond_name}]: {mean_eer:.2f}%')

        # Save AFTER EVERY CONDITION to prevent data loss on timeout
        out = Path(f'/kaggle/working/step3_extended_eval.json')
        with open(out, 'w') as f: json.dump(results, f, indent=2)

print('\n✓ Evaluation complete')

In [ ]:
# ── Summary Table ─────────────────────────────────────────────
conds = list(EVAL_CONDITIONS.keys())
vocs  = sorted(vocoder_dirs.keys())

for model_name in results['models']:
    print(f'\n{model_name} — WaveFake EER (%) per Vocoder x Codec Condition')
    hdr = f'  {"Vocoder":<32}' + ''.join(f'{c:>10}' for c in conds)
    print(hdr)
    print('  ' + '-' * (len(hdr) - 2))
    for voc in vocs:
        row = f'  {voc:<32}'
        for cond in conds:
            val = results['models'][model_name].get(cond, {}).get(voc)
            row += f'{val:>9.2f}%' if val is not None else f'{"N/A":>10}'
        print(row)
    avgs = [np.mean([results['models'][model_name].get(c, {}).get(v, 0) for v in vocs]) for c in conds]
    print(f'  {"Mean":<32}' + ''.join(f'{a:>9.2f}%' for a in avgs))

out = Path('/kaggle/working/step3_extended_eval.json')
with open(out, 'w') as f: json.dump(results, f, indent=2)
print(f'\n✓ Final results saved → {out}')
print('► Share this JSON with team — needed for Step 6 results table')